In [ ]:
!pip install langchain faiss-cpu tiktoken pypdf langchain-community langchain_openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 43.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 329.5/329.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 43.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.3/84.3 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


In [ ]:
!pip install -U langchain

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 102.1/102.1 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: langchain
    Found existing installation: langchain 1.1.0
    Uninstalling langchain-1.1.0:
      Successfully uninstalled langchain-1.1.0


In [ ]:
!pip show pypdf

Name: pypdf
Version: 6.4.0
Summary: A pure-python PDF library capable of splitting, merging, cropping, and transforming PDF files
Home-page: 
Author: 
Author-email: Mathieu Fenniak <biziqe@mathieu.fenniak.net>
License: 
Location: /usr/local/lib/python3.12/dist-packages
Requires: 
Required-by: 


In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_openai.llms import OpenAI

In [ ]:
#Load PDF
pdf_path = "/content/doc202517482201.pdf"
loader = PyPDFLoader(pdf_path)
pages = loader.load()
print(f"Loaded {len(pages)} pages")

Loaded 10 pages


In [ ]:
#split into chunks
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
chunks = splitter.split_documents(pages)
print(f"Split into {len(chunks)} chunks")
print("\n Sample Chunk \n",chunks[16].page_content[:500])

Split into 47 chunks

 Sample Chunk 
 Cement and Consumption of Steel, (vii) Net Tonne Kilometres and Passenger Kilometres  for 
Railways, (viii) Passenger and Cargo traffic handled by Civil Aviation, (ix) Cargo traffic handled at 
Major and Minor Sea Ports, (x) Sales of Commercial Vehicles, (xi) Bank Deposits and Credits, (xii) 
Premium related information of Life and Non -Life Insurance companies, (xiii) Data on outward 
Supplies of Goods and Services available from GSTN  upto November 2024 , (xiii) Accounts of


In [ ]:
embeddings = OpenAIEmbeddings()
vectorstore = FAISS.from_documents(chunks, embeddings)
print("Vectorstore created")

Vectorstore created


In [ ]:
query = "What is the GDP growth rate for India fro 2024-25?"
client = OpenAI()

In [ ]:
results = vectorstore.similarity_search(query,k=3)
context = "\n\n".join([doc.page_content for doc in results])
prompt=f'''
Use the following document context to answer the question.
Context: {context}
Question: {query}
Answer:
'''
rag_response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": prompt}
    ]
)
print("\n GDP with RAG")
print(rag_response.choices[0].message.content.strip())


 GDP with RAG
The GDP growth rate for India for 2024-25 is estimated at 6.4% based on the information provided in the document context.


In [ ]:
import tiktoken
def count_tokens(text:str,model:str="gpt-3.5-turbo"):
    encoding = tiktoken.encoding_for_model(model)
    num_tokens = len(encoding.encode(text))
    return num_tokens

In [ ]:
prompt=f'''
Use the following document context to answer the question.
Context: {context}
Question: {query}
Answer:
'''
rag_tokens = count_tokens(prompt)
print(f"Number of tokens in prompt: {rag_tokens}")
rag_cost = (rag_tokens/1000)*0.002
print(f"Cost of prompt: {rag_cost}")

Number of tokens in prompt: 433
Cost of prompt: 0.000866


In [ ]:
#without using RAG
from openai import OpenAI
client = OpenAI()
query = "What is the GDP growth rate for India fro 2024-25?"
direct_response = client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": query}
    ]
)
print("GDP without RAG")
print(direct_response.choices[0].message.content.strip())

GDP without RAG
I'm sorry, but I am unable to provide real-time or future data such as the GDP growth rate for India in 2024-25. I recommend checking official sources such as the World Bank, International Monetary Fund (IMF), or the Reserve Bank of India for the most accurate and up-to-date information on GDP growth rate projections.
